# RESEARCH NOTEBOOK --> SmugPlug

In [1]:
import os
import sys
import warnings
import asyncio
import aiohttp
import pandas as pd
import plotly.graph_objects as go
from datetime import datetime
import time
from typing import Dict, Optional, Tuple

warnings.filterwarnings("ignore")

root_path = os.path.abspath(os.path.join(os.getcwd(), '../..'))
sys.path.append(root_path)

In [2]:
import pandas as pd
import pandas_ta as ta  # noqa: F401
from core.data_sources import CLOBDataSource

# Initialize the data source
clob = CLOBDataSource()

In [3]:
# Define the parameters
exchange = "binance_perpetual"
trading_pair = "WLD-USDT"
timeframe = "1h"
days = 30

In [4]:
# Get the candles
candles = await clob.get_candles_last_days(
    exchange, trading_pair, timeframe, days, from_trades=False)

2025-02-04 22:04:08,243 - asyncio - ERROR - Unclosed client session
client_session: <aiohttp.client.ClientSession object at 0x168725c60>
2025-02-04 22:04:08,245 - asyncio - ERROR - Unclosed connector
connections: ['deque([(<aiohttp.client_proto.ResponseHandler object at 0x111a8a500>, 426878.057271291)])']
connector: <aiohttp.connector.TCPConnector object at 0x168725c90>


In [5]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [6]:
async def get_orderbook_data():
    connector = None
    try:
        # Get Binance order book through connector
        connector = clob.get_connector("binance_perpetual")
        orderbook = await connector._orderbook_ds._order_book_snapshot(trading_pair)
        
        # Create lists to store the data
        data = []
        
        # Add bids
        for price, amount, _ in orderbook.bids[:20]:  # Note: using all 3 values
            data.append({
                'price': float(price),
                'amount': float(amount),
                'side': 'bid'
            })
            
        # Add asks
        for price, amount, _ in orderbook.asks[:20]:  # Note: using all 3 values
            data.append({
                'price': float(price),
                'amount': float(amount),
                'side': 'ask'
            })
        
        # Create DataFrame from the list of dictionaries
        orderbook_df = pd.DataFrame(data)
        
        # Calculate cumulative amounts
        orderbook_df['cumulative_amount'] = orderbook_df.groupby('side')['amount'].cumsum()
        
        return orderbook_df
        
    except Exception as e:
        print(f"Error fetching order book data: {str(e)}")
        return None
    finally:
        # Properly close the connector and session
        if connector and hasattr(connector, '_session') and not connector._session.closed:
            await connector._session.close()
        if hasattr(clob, '_session') and not clob._session.closed:
            await clob._session.close()

# Get and display the order book data
orderbook_df = await get_orderbook_data()

if orderbook_df is not None:
    print("\nOrder Book Data:")
    print(orderbook_df)
    
    # You can also create a more detailed view of market depth
    print("\nMarket Depth Analysis:")
    print("\nBids Summary:")
    bids = orderbook_df[orderbook_df['side'] == 'bid'].head()
    print(bids)
    
    print("\nAsks Summary:")
    asks = orderbook_df[orderbook_df['side'] == 'ask'].head()
    print(asks)
    
    # Calculate and print some market metrics
    best_bid = bids['price'].iloc[0]
    best_ask = asks['price'].iloc[0]
    spread = best_ask - best_bid
    spread_pct = (spread / best_bid) * 100
    
    print(f"\nMarket Metrics:")
    print(f"Best Bid: {best_bid:.8f}")
    print(f"Best Ask: {best_ask:.8f}")
    print(f"Spread: {spread:.8f} ({spread_pct:.4f}%)")
    
    # Create a visualization of the order book
    fig = go.Figure()
    
    # Add bids
    bids = orderbook_df[orderbook_df['side'] == 'bid'].sort_values('price', ascending=True)
    asks = orderbook_df[orderbook_df['side'] == 'ask'].sort_values('price', ascending=True)
    
    fig.add_trace(go.Scatter(
        x=bids['price'],
        y=bids['cumulative_amount'],
        name='Bids',
        line=dict(color='green'),
        fill='tozeroy'
    ))
    
    # Add asks
    fig.add_trace(go.Scatter(
        x=asks['price'],
        y=asks['cumulative_amount'],
        name='Asks',
        line=dict(color='red'),
        fill='tozeroy'
    ))
    
    fig.update_layout(
        title=f'Order Book Depth - {trading_pair}',
        xaxis_title='Price',
        yaxis_title='Cumulative Amount',
        showlegend=True,
        template='plotly_dark'  # Using dark theme to match your other plots
    )
    
    fig.show()

2025-02-04 22:04:09,584 - asyncio - ERROR - Unclosed client session
client_session: <aiohttp.client.ClientSession object at 0x16915d5a0>
2025-02-04 22:04:09,590 - asyncio - ERROR - Unclosed connector
connections: ['deque([(<aiohttp.client_proto.ResponseHandler object at 0x111a8a500>, 426879.401836375)])']
connector: <aiohttp.connector.TCPConnector object at 0x16915d540>



Order Book Data:
    price  amount side  cumulative_amount
0  1.3287    2494  bid               2494
1  1.3286    1414  bid               3908
2  1.3285    2332  bid               6240
3  1.3284    3201  bid               9441
4  1.3283    7083  bid              16524
5  1.3282    4911  bid              21435
6  1.3281    3740  bid              25175
7   1.328    3431  bid              28606
8  1.3279    3931  bid              32537
9  1.3278    6481  bid              39018
10 1.3277    9892  bid              48910
11 1.3276   12526  bid              61436
12 1.3275    5796  bid              67232
13 1.3274    4089  bid              71321
14 1.3273    4799  bid              76120
15 1.3272    9091  bid              85211
16 1.3271    4339  bid              89550
17  1.327    6460  bid              96010
18 1.3269    5613  bid             101623
19 1.3268    9885  bid             111508
20 1.3288      47  ask                 47
21 1.3289      18  ask                 65
22  1.329     69

In [7]:
async def get_orderbook_data():
    connector = None
    try:
        # Get Binance order book through connector
        connector = clob.get_connector("okx_perpetual")
        orderbook = await connector._orderbook_ds._order_book_snapshot(trading_pair)
        
        # Create lists to store the data
        data = []
        
        # Add bids
        for price, amount, _ in orderbook.bids[:20]:  # Note: using all 3 values
            data.append({
                'price': float(price),
                'amount': float(amount),
                'side': 'bid'
            })
            
        # Add asks
        for price, amount, _ in orderbook.asks[:20]:  # Note: using all 3 values
            data.append({
                'price': float(price),
                'amount': float(amount),
                'side': 'ask'
            })
        
        # Create DataFrame from the list of dictionaries
        orderbook_df = pd.DataFrame(data)
        
        # Calculate cumulative amounts
        orderbook_df['cumulative_amount'] = orderbook_df.groupby('side')['amount'].cumsum()
        
        return orderbook_df
        
    except Exception as e:
        print(f"Error fetching order book data: {str(e)}")
        return None
    finally:
        # Properly close the connector and session
        if connector and hasattr(connector, '_session') and not connector._session.closed:
            await connector._session.close()
        if hasattr(clob, '_session') and not clob._session.closed:
            await clob._session.close()

# Get and display the order book data
orderbook_df = await get_orderbook_data()

if orderbook_df is not None:
    print("\nOrder Book Data:")
    print(orderbook_df)
    
    # You can also create a more detailed view of market depth
    print("\nMarket Depth Analysis:")
    print("\nBids Summary:")
    bids = orderbook_df[orderbook_df['side'] == 'bid'].head()
    print(bids)
    
    print("\nAsks Summary:")
    asks = orderbook_df[orderbook_df['side'] == 'ask'].head()
    print(asks)
    
    # Calculate and print some market metrics
    best_bid = bids['price'].iloc[0]
    best_ask = asks['price'].iloc[0]
    spread = best_ask - best_bid
    spread_pct = (spread / best_bid) * 100
    
    print(f"\nMarket Metrics:")
    print(f"Best Bid: {best_bid:.8f}")
    print(f"Best Ask: {best_ask:.8f}")
    print(f"Spread: {spread:.8f} ({spread_pct:.4f}%)")
    
    # Create a visualization of the order book
    fig = go.Figure()
    
    # Add bids
    bids = orderbook_df[orderbook_df['side'] == 'bid'].sort_values('price', ascending=True)
    asks = orderbook_df[orderbook_df['side'] == 'ask'].sort_values('price', ascending=True)
    
    fig.add_trace(go.Scatter(
        x=bids['price'],
        y=bids['cumulative_amount'],
        name='Bids',
        line=dict(color='green'),
        fill='tozeroy'
    ))
    
    # Add asks
    fig.add_trace(go.Scatter(
        x=asks['price'],
        y=asks['cumulative_amount'],
        name='Asks',
        line=dict(color='red'),
        fill='tozeroy'
    ))
    
    fig.update_layout(
        title=f'Order Book Depth - {trading_pair}',
        xaxis_title='Price',
        yaxis_title='Cumulative Amount',
        showlegend=True,
        template='plotly_dark'  # Using dark theme to match your other plots
    )
    
    fig.show()

2025-02-04 22:04:14,747 - asyncio - ERROR - Unclosed client session
client_session: <aiohttp.client.ClientSession object at 0x169163b80>
2025-02-04 22:04:14,749 - asyncio - ERROR - Unclosed connector
connections: ['deque([(<aiohttp.client_proto.ResponseHandler object at 0x168750340>, 426884.565977791)])']
connector: <aiohttp.connector.TCPConnector object at 0x169163ac0>



Order Book Data:
    price  amount side  cumulative_amount
0    1.33   11776  bid              11776
1   1.329   42277  bid              54053
2   1.328   48967  bid             103020
3   1.327   37606  bid             140626
4   1.326   55001  bid             195627
5   1.325   96080  bid             291707
6   1.324   46795  bid             338502
7   1.323   17311  bid             355813
8   1.322  127184  bid             482997
9   1.321   32255  bid             515252
10   1.32   34235  bid             549487
11  1.319   81897  bid             631384
12  1.318   58901  bid             690285
13  1.317   40193  bid             730478
14  1.316  130121  bid             860599
15  1.315   33067  bid             893666
16  1.314   47447  bid             941113
17  1.313   25012  bid             966125
18  1.312   19575  bid             985700
19  1.311   35631  bid            1021331
20  1.331   19042  ask              19042
21  1.332  106742  ask             125784
22  1.333   9201

In [8]:
async def get_orderbook_data():
    connector = None
    try:
        # Get Binance order book through connector
        connector = clob.get_connector("kucoin")
        orderbook = await connector._orderbook_ds._order_book_snapshot(trading_pair)
        
        # Create lists to store the data
        data = []
        
        # Add bids
        for price, amount, _ in orderbook.bids[:20]:  # Note: using all 3 values
            data.append({
                'price': float(price),
                'amount': float(amount),
                'side': 'bid'
            })
            
        # Add asks
        for price, amount, _ in orderbook.asks[:20]:  # Note: using all 3 values
            data.append({
                'price': float(price),
                'amount': float(amount),
                'side': 'ask'
            })
        
        # Create DataFrame from the list of dictionaries
        orderbook_df = pd.DataFrame(data)
        
        # Calculate cumulative amounts
        orderbook_df['cumulative_amount'] = orderbook_df.groupby('side')['amount'].cumsum()
        
        return orderbook_df
        
    except Exception as e:
        print(f"Error fetching order book data: {str(e)}")
        return None
    finally:
        # Properly close the connector and session
        if connector and hasattr(connector, '_session') and not connector._session.closed:
            await connector._session.close()
        if hasattr(clob, '_session') and not clob._session.closed:
            await clob._session.close()

# Get and display the order book data
orderbook_df = await get_orderbook_data()

if orderbook_df is not None:
    print("\nOrder Book Data:")
    print(orderbook_df)
    
    # You can also create a more detailed view of market depth
    print("\nMarket Depth Analysis:")
    print("\nBids Summary:")
    bids = orderbook_df[orderbook_df['side'] == 'bid'].head()
    print(bids)
    
    print("\nAsks Summary:")
    asks = orderbook_df[orderbook_df['side'] == 'ask'].head()
    print(asks)
    
    # Calculate and print some market metrics
    best_bid = bids['price'].iloc[0]
    best_ask = asks['price'].iloc[0]
    spread = best_ask - best_bid
    spread_pct = (spread / best_bid) * 100
    
    print(f"\nMarket Metrics:")
    print(f"Best Bid: {best_bid:.8f}")
    print(f"Best Ask: {best_ask:.8f}")
    print(f"Spread: {spread:.8f} ({spread_pct:.4f}%)")
    
    # Create a visualization of the order book
    fig = go.Figure()
    
    # Add bids
    bids = orderbook_df[orderbook_df['side'] == 'bid'].sort_values('price', ascending=True)
    asks = orderbook_df[orderbook_df['side'] == 'ask'].sort_values('price', ascending=True)
    
    fig.add_trace(go.Scatter(
        x=bids['price'],
        y=bids['cumulative_amount'],
        name='Bids',
        line=dict(color='green'),
        fill='tozeroy'
    ))
    
    # Add asks
    fig.add_trace(go.Scatter(
        x=asks['price'],
        y=asks['cumulative_amount'],
        name='Asks',
        line=dict(color='red'),
        fill='tozeroy'
    ))
    
    fig.update_layout(
        title=f'Order Book Depth - {trading_pair}',
        xaxis_title='Price',
        yaxis_title='Cumulative Amount',
        showlegend=True,
        template='plotly_dark'  # Using dark theme to match your other plots
    )
    
    fig.show()

2025-02-04 22:04:18,953 - asyncio - ERROR - Unclosed client session
client_session: <aiohttp.client.ClientSession object at 0x16915e770>
2025-02-04 22:04:18,954 - asyncio - ERROR - Unclosed connector
connections: ['deque([(<aiohttp.client_proto.ResponseHandler object at 0x1691253c0>, 426888.7716365)])']
connector: <aiohttp.connector.TCPConnector object at 0x16915e6e0>



Order Book Data:
    price    amount side  cumulative_amount
0  1.3303    95.375  bid             95.375
1  1.3302    39.375  bid             134.75
2  1.3301    1.4477  bid           136.1977
3    1.33  728.0314  bid           864.2291
4  1.3299 1049.6015  bid          1913.8306
5  1.3297  117.2036  bid          2031.0342
6  1.3296  448.9204  bid          2479.9546
7  1.3295  811.0572  bid          3291.0118
8  1.3294 1380.8146  bid          4671.8264
9  1.3293  819.0413  bid          5490.8677
10 1.3292  373.7759  bid          5864.6436
11 1.3291 1167.7388  bid          7032.3824
12  1.329 3761.2125  bid         10793.5949
13 1.3289    39.375  bid         10832.9699
14 1.3288  373.7759  bid         11206.7458
15 1.3287    39.375  bid         11246.1208
16 1.3286  124.4203  bid         11370.5411
17 1.3285  273.2749  bid          11643.816
18 1.3284 1166.0501  bid         12809.8661
19 1.3283 1181.0609  bid          13990.927
20 1.3306    39.375  ask             39.375
21 1.3307   14

In [9]:
candles.data

,timestamp,open,high,low,close,volume,quote_asset_volume,n_trades,taker_buy_base_volume,taker_buy_quote_volume
timestamp,,,,,,,,,,
2025-01-06 04:00:00,1736136000,2.4401,2.4524,2.4127,2.4305,4450362,10806008.2487,43446,2148963,5217378.1832
2025-01-06 05:00:00,1736139600,2.4304,2.4598,2.4212,2.4405,5851437,14292148.5297,52796,3173089,7746454.4195
2025-01-06 06:00:00,1736143200,2.4405,2.4949,2.4325,2.4649,8506479,20991810.3538,71329,4609983,11376262.4994
2025-01-06 07:00:00,1736146800,2.4649,2.4836,2.4403,2.4548,6285144,15455882.136,60146,3034097,7462626.215
2025-01-06 08:00:00,1736150400,2.4549,2.46,2.4131,2.4321,6565824,15968486.2662,56372,3170523,7713168.718
...,...,...,...,...,...,...,...,...,...,...
2025-02-04 23:00:00,1738710000,1.3021,1.3269,1.2935,1.3139,7214662,9451311.5682,53816,3716270,4868997.8128
2025-02-05 00:00:00,1738713600,1.314,1.3225,1.2985,1.3088,7411912,9709935.6994,56849,3459856,4534240.7804
2025-02-05 01:00:00,1738717200,1.3086,1.3297,1.3079,1.3131,6657405,8789708.7688,47945,3585990,4735458.3331


In [10]:
candles.plot(type="returns", height=400, width=800)

2025-02-04 22:04:21,216 - asyncio - ERROR - Unclosed client session
client_session: <aiohttp.client.ClientSession object at 0x169160130>
2025-02-04 22:04:21,218 - asyncio - ERROR - Unclosed connector
connections: ['deque([(<aiohttp.client_proto.ResponseHandler object at 0x169d0ebc0>, 426887.852276375)])']
connector: <aiohttp.connector.TCPConnector object at 0x169160190>
2025-02-04 22:04:21,219 - asyncio - ERROR - Unclosed client session
client_session: <aiohttp.client.ClientSession object at 0x16913e7a0>
2025-02-04 22:04:21,220 - asyncio - ERROR - Unclosed connector
connections: ['deque([(<aiohttp.client_proto.ResponseHandler object at 0x169d0dc00>, 426890.894520333)])']
connector: <aiohttp.connector.TCPConnector object at 0x16910b3a0>


In [11]:
# EMAs
ema_short = 8
ema_medium = 29
ema_long = 31

# MACD
macd_fast = 22
macd_slow = 36
macd_signal = 17

# ATR
atr_length = 3
atr_multiplier = 1.5

# Add indicators
candles_df = candles.data
candles_df.ta.macd(fast=macd_fast, slow=macd_slow, signal=macd_signal, append=True)
candles_df.ta.atr(length=atr_length, append=True)
candles_df.ta.ema(length=ema_short, append=True)
candles_df.ta.ema(length=ema_medium, append=True)
candles_df.ta.ema(length=ema_long, append=True)
candles_df["long_atr_support"] = candles_df["close"].shift(1) - candles_df[f"ATRr_{atr_length}"] * atr_multiplier
candles_df["short_atr_resistance"] = candles_df["close"].shift(1) + candles_df[f"ATRr_{atr_length}"] * atr_multiplier

candles_df.tail(5)

,timestamp,open,high,low,close,volume,quote_asset_volume,n_trades,taker_buy_base_volume,taker_buy_quote_volume,MACD_22_36_17,MACDh_22_36_17,MACDs_22_36_17,ATRr_11,EMA_8,EMA_29,EMA_31,long_atr_support,short_atr_resistance
timestamp,,,,,,,,,,,,,,,,,,,
2025-02-04 23:00:00,1738710000,1.3021,1.3269,1.2935,1.3139,7214662,9451311.5682,53816,3716270,4868997.8128,-0.01254788,0.00466153,-0.01720941,0.04192801,1.30957198,1.32319587,1.32509048,1.23920799,1.36499201
2025-02-05 00:00:00,1738713600,1.314,1.3225,1.2985,1.3088,7411912,9709935.6994,56849,3459856,4534240.7804,-0.01218055,0.0044701,-0.01665065,0.04029819,1.30940043,1.32223615,1.32407232,1.25345272,1.37434728
2025-02-05 01:00:00,1738717200,1.3086,1.3297,1.3079,1.3131,6657405,8789708.7688,47945,3585990,4735458.3331,-0.01166455,0.00443209,-0.01609664,0.03861653,1.31022256,1.32162707,1.32338655,1.2508752,1.3667248
2025-02-05 02:00:00,1738720800,1.3132,1.3424,1.3121,1.3332,8915501,11814143.1811,65417,4683056,6207320.3539,-0.01050273,0.00497237,-0.01547509,0.03786048,1.31532865,1.3223986,1.32399989,1.25630927,1.36989073
2025-02-05 03:00:00,1738724400,1.3332,1.3375,1.3288,1.3288,399009,531655.8998,3817,211004,281185.9173,-0.00959467,0.00522704,-0.01482171,0.03520953,1.31832229,1.32282536,1.3242999,1.2803857,1.3860143


In [12]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Create figure with secondary y-axis
fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.03, 
                    subplot_titles=(trading_pair, 'MACD'),
                    row_heights=[0.7, 0.3])

# Add candlestick
fig.add_trace(go.Candlestick(x=candles_df.index,
                             open=candles_df['open'],
                             high=candles_df['high'],
                             low=candles_df['low'],
                             close=candles_df['close'],
                             name='OHLC'),
              row=1, col=1)

# Add EMAs
ema_fast = f'EMA_{ema_short}'
ema_med = f'EMA_{ema_medium}' 
ema_slow = f'EMA_{ema_long}'

fig.add_trace(go.Scatter(x=candles_df.index, y=candles_df[ema_fast],
                         line=dict(color='#00FF00', width=2),
                         name='Fast EMA'), row=1, col=1)
fig.add_trace(go.Scatter(x=candles_df.index, y=candles_df[ema_med],
                         line=dict(color='#FFA500', width=2), 
                         name='Medium EMA'), row=1, col=1)
fig.add_trace(go.Scatter(x=candles_df.index, y=candles_df[ema_slow],
                         line=dict(color='#0000FF', width=2),
                         name='Slow EMA'), row=1, col=1)

# Add MACD
macd = f'MACD_{macd_fast}_{macd_slow}_{macd_signal}'
macd_s = f'MACDs_{macd_fast}_{macd_slow}_{macd_signal}'
macd_hist = f'MACDh_{macd_fast}_{macd_slow}_{macd_signal}'

fig.add_trace(go.Scatter(x=candles_df.index, y=candles_df[macd], 
                         line=dict(color='#00FFFF', width=2),
                         name='MACD'), row=2, col=1)
fig.add_trace(go.Scatter(x=candles_df.index, y=candles_df[macd_s], 
                         line=dict(color='#FFA500', width=2),
                         name='Signal'), row=2, col=1)
fig.add_trace(go.Bar(x=candles_df.index, y=candles_df[macd_hist], name='Histogram',
                     marker_color=candles_df[macd_hist].apply(
                         lambda x: '#00FF00' if x >= 0 else '#FF0000')),
                    row=2, col=1)

# Add ATR support and resistance
fig.add_trace(go.Scatter(x=candles_df.index, y=candles_df["long_atr_support"],
                         line=dict(color='#00FF00', width=2),
                         name='Long ATR Support'), row=1, col=1)
fig.add_trace(go.Scatter(x=candles_df.index, y=candles_df["short_atr_resistance"],
                         line=dict(color='#FF0000', width=2),
                         name='Short ATR Resistance'), row=1, col=1)

# Update layout for dark theme
fig.update_layout(
    title=f'{exchange} - {trading_pair} - {timeframe}',
    width=1200, height=800,
    font=dict(color='#e1e1e1'),
    plot_bgcolor='#1e1e1e',
    paper_bgcolor='#1e1e1e',
    xaxis_rangeslider_visible=False,
    legend=dict(bgcolor='rgba(0,0,0,0)'),
    yaxis=dict(title='Price'),
    yaxis2=dict(title='MACD', showgrid=False),
    showlegend=False
)

# Update axes
fig.update_xaxes(showgrid=True, gridwidth=1, gridcolor='#323232', zeroline=False)
fig.update_yaxes(showgrid=True, gridwidth=1, gridcolor='#323232', zeroline=False)

# Show the plot
fig.show()

In [13]:
# Generate signal


macdh = candles_df[f"MACDh_{macd_fast}_{macd_slow}_{macd_signal}"]
short_ema = candles_df[f"EMA_{ema_short}"]
medium_ema = candles_df[f"EMA_{ema_medium}"]
long_ema = candles_df[f"EMA_{ema_long}"]
close = candles_df["close"]


long_condition = (short_ema > medium_ema) & (medium_ema > long_ema) & (close > short_ema) & (close > candles_df["long_atr_support"]) & (macdh > 0) 
short_condition = (short_ema < medium_ema) & (medium_ema < long_ema) & (close < short_ema) & (close < candles_df["short_atr_resistance"]) & (macdh < 0)

candles_df["signal"] = 0
candles_df.loc[long_condition, "signal"] = 1
candles_df.loc[short_condition, "signal"] = -1

In [14]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go

fig = make_subplots(rows=3, cols=1, shared_xaxes=True, vertical_spacing=0.02,
                    subplot_titles=('OHLC with BB', 'MACD', 'Signal'),
                    row_heights=[0.6, 0.2, 0.2])

# Add candlestick
fig.add_trace(go.Candlestick(x=candles_df.index,
                             open=candles_df['open'],
                             high=candles_df['high'],
                             low=candles_df['low'],
                             close=candles_df['close'],
                             name='Candlesticks'),
              row=1, col=1)


# Add EMAs
ema_fast = f'EMA_{ema_short}'
ema_med = f'EMA_{ema_medium}' 
ema_slow = f'EMA_{ema_long}'

fig.add_trace(go.Scatter(x=candles_df.index, y=candles_df[ema_fast],
                         line=dict(color='#00FF00', width=2),
                         name='Fast EMA'), row=1, col=1)
fig.add_trace(go.Scatter(x=candles_df.index, y=candles_df[ema_med],
                         line=dict(color='#FFA500', width=2), 
                         name='Medium EMA'), row=1, col=1)
fig.add_trace(go.Scatter(x=candles_df.index, y=candles_df[ema_slow],
                         line=dict(color='#0000FF', width=2),
                         name='Slow EMA'), row=1, col=1)

# Add MACD
macd = f'MACD_{macd_fast}_{macd_slow}_{macd_signal}'
macd_s = f'MACDs_{macd_fast}_{macd_slow}_{macd_signal}'
macd_hist = f'MACDh_{macd_fast}_{macd_slow}_{macd_signal}'

fig.add_trace(go.Scatter(x=candles_df.index, y=candles_df[macd], 
                         line=dict(color='#00FFFF', width=2),
                         name='MACD'), row=2, col=1)
fig.add_trace(go.Scatter(x=candles_df.index, y=candles_df[macd_s], 
                         line=dict(color='#FFA500', width=2),
                         name='Signal'), row=2, col=1)
fig.add_trace(go.Bar(x=candles_df.index, y=candles_df[macd_hist], name='Histogram',
                     marker_color=candles_df[macd_hist].apply(
                         lambda x: '#00FF00' if x >= 0 else '#FF0000')),
              row=2, col=1)

# Add ATR support and resistance
fig.add_trace(go.Scatter(x=candles_df.index, y=candles_df["long_atr_support"],
                         line=dict(color='#00FF00', width=2),
                         name='Long ATR Support'), row=1, col=1)
fig.add_trace(go.Scatter(x=candles_df.index, y=candles_df["short_atr_resistance"],
                         line=dict(color='#FF0000', width=2),
                         name='Short ATR Resistance'), row=1, col=1)

# Add the signal line
fig.add_trace(go.Scatter(x=candles_df.index, y=candles_df['signal'],
                         mode='lines',
                         name='Signal',
                         line=dict(color="white")),
              row=3, col=1)

# Update layout for dark theme
fig.update_layout(
    title=f'{exchange} - {trading_pair} - {timeframe}',
    width=1500, height=1000,
    font=dict(color='#e1e1e1'),
    plot_bgcolor='#1e1e1e',
    paper_bgcolor='#1e1e1e',
    xaxis_rangeslider_visible=False,
    legend=dict(bgcolor='rgba(0,0,0,0)'),
    yaxis=dict(title='Price'),
    yaxis2=dict(title='MACD', showgrid=False),
    yaxis3=dict(title='Signal', showgrid=False),
    showlegend=False
)

# Update axes
fig.update_xaxes(showgrid=True, gridwidth=1, gridcolor='#323232', zeroline=False)
fig.update_yaxes(showgrid=True, gridwidth=1, gridcolor='#323232', zeroline=False)

# Show the plot
fig.show()


# CONCLUSION

In this notebook, we have implemented a strategy combining the MACD (Moving Average Convergence Divergence) indicator with Bollinger Bands. We've visualized these indicators along with the price data and generated signals based on their interactions. This approach provides a solid foundation for our trading strategy.
 
## Key components of our strategy include:
 1. MACD for trend identification
 2. Bollinger Bands for volatility measurement and potential reversal points
 3. A signal line derived from the combination of these indicators
 
 The next step is to backtest this strategy to evaluate its profitability and robustness. For this purpose, we have created a controller file named `macd_bb.py` in this folder. This file implements the logic we've developed here, allowing us to conduct comprehensive backtests in the subsequent notebook.